# 02 — Train DenseNet121 (5-Fold Cross-Validation)

This notebook trains DenseNet121 using a two-phase strategy:
1. **Head Training** — Base frozen, new classification head trained.
2. **Fine-Tuning** — Full model unfrozen, trained with a very low learning rate.

> **Educational Use Only** — Not a medical diagnostic tool.

## Step 1 — Verify GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
if not gpus:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')
else:
    for gpu in gpus:
        print(f'  {gpu.name}')

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3 — Clone or Update the Repository

In [ ]:
import os

# --- Configure these paths to match your Google Drive layout ---
DRIVE_BASE = '/content/drive/MyDrive'
REPO_DIR = os.path.join(DRIVE_BASE, 'histology-ai-classification')
REPO_URL = 'https://github.com/<your-org>/histology-ai-classification.git'

if os.path.exists(REPO_DIR):
    print('Repository exists — pulling latest...')
    os.chdir(REPO_DIR)
    !git pull
else:
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print('Working directory:', os.getcwd())

## Step 4 — Install Dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

## Step 5 — Configure Training Parameters

In [ ]:
# --- Adjust these settings as needed ---
CONFIG_PATH = 'configs/densenet121.yaml'
OUTPUT_DIR  = os.path.join(DRIVE_BASE, 'histology-results')

# Set to None to train all 5 folds, or set to a specific fold index (0-4)
TRAIN_SINGLE_FOLD = None  # e.g. 0 for a quick test

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Config:     {CONFIG_PATH}')
print(f'Output dir: {OUTPUT_DIR}')

## Step 6 — Load Configuration

In [ ]:
from src.utils.config import load_yaml
config = load_yaml(CONFIG_PATH)
print('Dataset root :', config['data']['dataset_root'])
print('Folds path   :', config['data']['folds_path'])
print('Num classes  :', config['data']['num_classes'])
print('Num folds    :', config['validation']['num_folds'])

## Step 7 — Run Training

In [ ]:
from src.training.train import run_cross_validation, train_fold
from pathlib import Path

if TRAIN_SINGLE_FOLD is not None:
    print(f'Training single fold: {TRAIN_SINGLE_FOLD}')
    metrics = train_fold(config, fold=TRAIN_SINGLE_FOLD, output_dir=Path(OUTPUT_DIR))
    print('Metrics:', metrics)
else:
    print('Running full 5-fold cross-validation...')
    run_cross_validation(CONFIG_PATH, OUTPUT_DIR)

## Step 8 — Review Results

In [ ]:
import json
from pathlib import Path

metrics_dir = Path(OUTPUT_DIR) / 'reports' / 'densenet121' / 'metrics'
summary_path = metrics_dir / 'cross_validation_summary.json'

if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print('Average Accuracy:', summary['average']['accuracy'])
    print('Average Macro F1:', summary['average']['macro_f1'])
else:
    print('Summary not yet available (single fold mode or training not complete).')
    for fold_file in sorted(metrics_dir.glob('fold_*.json')):
        with open(fold_file) as f:
            m = json.load(f)
        print(f'{fold_file.stem}: accuracy={m["accuracy"]:.4f}, macro_f1={m["macro_f1"]:.4f}')